# ML Pipeline (dev notebook)
**Purpose:** run the ML steps quickly during development, without waiting for Airflow.
Calls the **same `utils/` functions** as the Airflow scripts, so results are the same. The final run is done by the Airflow DAG.

Steps:
1. (optional) rebuild datamart locally
2. Train models and save best to model bank
3. Inspect model artefact

In [4]:
import os
import pickle
import pprint
import pyspark
import pandas as pd

import utils.model_train

## Setup Spark

In [5]:
# set path for Hadoop (Windows only, comment out on Mac/Linux)
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = os.environ["PATH"] + ";C:\\hadoop\\bin"

# tell Spark which python to use for its python workers (Windows has no 'python3', which is Spark's default)
import sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

# use Arrow to convert pandas <-> Spark (python workers crash on Windows when converting pandas to Spark)
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

## 1. (optional) Rebuild datamart locally
Only needed if `datamart/` is missing or utils changed. Runs bronze → silver → gold for all months (a few minutes, much faster than Airflow because Spark starts only once).

In [3]:
# %run main.py

## 2. Train models
Training date 2024-06-01: train/test 2023-01..2023-10, OOT 2023-11..2023-12 (only labels observed by training date).

In [4]:
model_train_date_str = "2024-06-01"

In [5]:
model_artefact = utils.model_train.train_model(
    model_train_date_str,
    gold_feature_store_directory="datamart/gold/feature_store/",
    gold_label_store_directory="datamart/gold/label_store/",
    model_bank_directory="model_bank/",
    spark=spark,
)

{'model_train_date': datetime.datetime(2024, 6, 1, 0, 0),
 'model_train_date_str': '2024-06-01',
 'oot_end_date': datetime.datetime(2023, 12, 1, 0, 0),
 'oot_start_date': datetime.datetime(2023, 11, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2023, 10, 1, 0, 0),
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 1, 1, 0, 0)}
extracted labels_sdf 5958
extracted features_sdf 5958
X_train 3982 | X_test 996 | X_oot 980 | features 62
bad rate train 0.28 | test 0.28 | oot 0.3
Fitting 3 folds for each of 50 candidates, totalling 150 fits
{'logistic_regression': {'auc_oot': np.float64(0.8724737708494477),
                         'auc_test': np.float64(0.8176442064956034),
                         'auc_train': np.float64(0.8554483171567553),
                         'gini_oot': np.float64(0.745),
                         'gini_test': np.float64(0.635),
                         'gini_train': np.float64(0.711)},
 'xgboost': {'auc_oot': np.float64(0.88420995220245

## 3. Inspect model artefact

In [6]:
file_path = "model_bank/credit_model_" + model_train_date_str.replace("-", "_") + ".pkl"
with open(file_path, "rb") as file:
    loaded_model_artefact = pickle.load(file)

print("model_version:", loaded_model_artefact["model_version"])
print("model_type:   ", loaded_model_artefact["model_type"])
print("n features:   ", len(loaded_model_artefact["feature_cols"]))
print("\ndata_stats:"); pprint.pprint(loaded_model_artefact["data_stats"])
print("\ncandidate_results:"); pprint.pprint(loaded_model_artefact["candidate_results"])
print("\ncalibration_results:"); pprint.pprint(loaded_model_artefact["calibration_results"])
print("\nhp_params:"); pprint.pprint(loaded_model_artefact["hp_params"])

model_version: credit_model_2024_06_01
model_type:    xgboost
n features:    62

data_stats:
{'X_oot': 980,
 'X_test': 996,
 'X_train': 3982,
 'y_oot': np.float64(0.3),
 'y_test': np.float64(0.28),
 'y_train': np.float64(0.28)}

candidate_results:
{'logistic_regression': {'auc_oot': np.float64(0.8724737708494477),
                         'auc_test': np.float64(0.8176442064956034),
                         'auc_train': np.float64(0.8554483171567553),
                         'gini_oot': np.float64(0.745),
                         'gini_test': np.float64(0.635),
                         'gini_train': np.float64(0.711)},
 'xgboost': {'auc_oot': np.float64(0.8842099522024554),
             'auc_test': np.float64(0.8387296731202791),
             'auc_train': np.float64(0.9080034836560351),
             'gini_oot': np.float64(0.768),
             'gini_test': np.float64(0.677),
             'gini_train': np.float64(0.816)}}

calibration_results:
{'auc_oot_calibrated': np.float64(0.88420995

## 4. Inference
Model deployed at 2024-06-01: score new applications every month from 2024-06 to 2025-01 (no applications after 2025-01, those months are skipped).
Output: `datamart/gold/model_predictions/credit_model_2024_06_01/`

In [7]:
import utils.model_inference
import pandas as pd

model_name = "credit_model_2024_06_01.pkl"
inference_dates = pd.date_range("2024-06-01", "2025-01-01", freq="MS").strftime("%Y-%m-%d").tolist()

for date_str in inference_dates:
    utils.model_inference.model_inference(
        date_str,
        model_bank_directory="model_bank/",
        model_name=model_name,
        gold_feature_store_directory="datamart/gold/feature_store/",
        gold_predictions_directory="datamart/gold/model_predictions/",
        spark=spark,
    )

Model loaded successfully! model_bank/credit_model_2024_06_01.pkl
extracted features 498 2024-06-01
saved to: datamart/gold/model_predictions/credit_model_2024_06_01/credit_model_2024_06_01_predictions_2024_06_01.parquet | rows: 498
Model loaded successfully! model_bank/credit_model_2024_06_01.pkl
extracted features 505 2024-07-01
saved to: datamart/gold/model_predictions/credit_model_2024_06_01/credit_model_2024_06_01_predictions_2024_07_01.parquet | rows: 505
Model loaded successfully! model_bank/credit_model_2024_06_01.pkl
extracted features 543 2024-08-01
saved to: datamart/gold/model_predictions/credit_model_2024_06_01/credit_model_2024_06_01_predictions_2024_08_01.parquet | rows: 543
Model loaded successfully! model_bank/credit_model_2024_06_01.pkl
extracted features 493 2024-09-01
saved to: datamart/gold/model_predictions/credit_model_2024_06_01/credit_model_2024_06_01_predictions_2024_09_01.parquet | rows: 493
Model loaded successfully! model_bank/credit_model_2024_06_01.pkl
ex

In [8]:
# check predictions gold table
import glob

pred_files = sorted(glob.glob("datamart/gold/model_predictions/credit_model_2024_06_01/*.parquet"))
df_pred = pd.concat([pd.read_parquet(f) for f in pred_files])
print("prediction partitions:", len(pred_files), "| rows:", len(df_pred))
print(df_pred.groupby("snapshot_date")["model_predictions"].agg(["count", "mean"]).round(3))
df_pred.head()

prediction partitions: 8 | rows: 4024
               count   mean
snapshot_date              
2024-06-01       498  0.288
2024-07-01       505  0.213
2024-08-01       543  0.208
2024-09-01       493  0.213
2024-10-01       456  0.192
2024-11-01       488  0.207
2024-12-01       515  0.180
2025-01-01       526  0.211


,loan_id,Customer_ID,snapshot_date,model_name,model_predictions
0,CUS_0x10dd_2024_06_01,CUS_0x10dd,2024-06-01,credit_model_2024_06_01,0.087493
1,CUS_0x1109_2024_06_01,CUS_0x1109,2024-06-01,credit_model_2024_06_01,0.606442
2,CUS_0x1286_2024_06_01,CUS_0x1286,2024-06-01,credit_model_2024_06_01,0.089159
3,CUS_0x12a8_2024_06_01,CUS_0x12a8,2024-06-01,credit_model_2024_06_01,0.081101
4,CUS_0x1309_2024_06_01,CUS_0x1309,2024-06-01,credit_model_2024_06_01,0.076973


## 5. Monitoring
Runs every month from 2024-06 to 2025-07:
- **stability** of predictions made that month (PSI on PD scores, CSI on top 5 features)
- **performance** of predictions made 6 months earlier, whose labels arrive that month (AUC, Gini, Brier, precision/recall/F1 at PD 0.5)

Output (long format, one row per metric): `datamart/gold/model_monitoring/credit_model_2024_06_01/`

In [9]:
import utils.model_monitoring

monitoring_dates = pd.date_range("2024-06-01", "2025-07-01", freq="MS").strftime("%Y-%m-%d").tolist()

for date_str in monitoring_dates:
    utils.model_monitoring.model_monitoring(
        date_str,
        model_bank_directory="model_bank/",
        model_name=model_name,
        gold_predictions_directory="datamart/gold/model_predictions/",
        gold_feature_store_directory="datamart/gold/feature_store/",
        gold_label_store_directory="datamart/gold/label_store/",
        gold_monitoring_directory="datamart/gold/model_monitoring/",
        spark=spark,
    )

2024-06-01 | stability: n=498 | psi=0.0459
2024-06-01 | performance: no predictions with labels arriving this month
saved to: datamart/gold/model_monitoring/credit_model_2024_06_01/credit_model_2024_06_01_monitoring_2024_06_01.parquet | rows: 8
2024-07-01 | stability: n=505 | psi=1.2138
2024-07-01 | performance: no predictions with labels arriving this month
saved to: datamart/gold/model_monitoring/credit_model_2024_06_01/credit_model_2024_06_01_monitoring_2024_07_01.parquet | rows: 8
2024-08-01 | stability: n=543 | psi=1.3198
2024-08-01 | performance: no predictions with labels arriving this month
saved to: datamart/gold/model_monitoring/credit_model_2024_06_01/credit_model_2024_06_01_monitoring_2024_08_01.parquet | rows: 8
2024-09-01 | stability: n=493 | psi=1.3301
2024-09-01 | performance: no predictions with labels arriving this month
saved to: datamart/gold/model_monitoring/credit_model_2024_06_01/credit_model_2024_06_01_monitoring_2024_09_01.parquet | rows: 8
2024-10-01 | stabili

In [10]:
# check monitoring gold table: one column per prediction month
mon_files = sorted(glob.glob("datamart/gold/model_monitoring/credit_model_2024_06_01/*.parquet"))
df_mon = pd.concat([pd.read_parquet(f) for f in mon_files])
print("monitoring partitions:", len(mon_files), "| rows:", len(df_mon))

print("\nstability (by prediction month):")
print(df_mon[df_mon.metric_type == "stability"].pivot(index="metric_name", columns="prediction_month", values="metric_value").round(3))

print("\nperformance (by prediction month, known 6 months later):")
print(df_mon[df_mon.metric_type == "performance"].pivot(index="metric_name", columns="prediction_month", values="metric_value").round(3))

print("\nalerts (amber / red):")
df_mon[df_mon.status.isin(["amber", "red"])][["prediction_month", "metric_name", "metric_value", "status"]]

monitoring partitions: 14 | rows: 144

stability (by prediction month):
prediction_month       2024-06-01  2024-07-01  2024-08-01  2024-09-01  \
metric_name                                                             
csi_Interest_Rate           0.017       0.027       0.037       0.043   
csi_Num_Bank_Accounts       0.028       0.010       0.023       0.017   
csi_Outstanding_Debt        0.012       0.019       0.006       0.017   
csi_fe_5_mean               0.779      16.110      16.110      16.110   
csi_num_fin_products        0.035       0.010       0.027       0.012   
mean_pd                     0.288       0.213       0.208       0.213   
n_predictions             498.000     505.000     543.000     493.000   
psi_score                   0.046       1.214       1.320       1.330   

prediction_month       2024-10-01  2024-11-01  2024-12-01  2025-01-01  
metric_name                                                            
csi_Interest_Rate           0.020       0.019       0

,prediction_month,metric_name,metric_value,status
7,2024-06-01,csi_fe_5_mean,0.779180,red
2,2024-07-01,psi_score,1.213757,red
7,2024-07-01,csi_fe_5_mean,16.110271,red
2,2024-08-01,psi_score,1.319780,red
7,2024-08-01,csi_fe_5_mean,16.110271,red
2,2024-09-01,psi_score,1.330115,red
7,2024-09-01,csi_fe_5_mean,16.110271,red
2,2024-10-01,psi_score,1.314933,red
7,2024-10-01,csi_fe_5_mean,16.110271,red
2,2024-11-01,psi_score,1.331465,red


## 6. Visualisation
Same function as the Airflow `model_visualisation` task: reads the monitoring gold table and saves PNG charts
(`psi.png`, `csi.png`, `performance_auc_gini.png`, `bad_rate.png`, `monitoring_dashboard.png`) to
`datamart/gold/model_monitoring_charts/credit_model_2024_06_01/`.

In [ ]:
import utils.model_visualisation
from IPython.display import Image, display

chart_files = utils.model_visualisation.plot_monitoring_charts(
    "2025-07-01",  # all results known by the end of the monitoring period
    model_name=model_name,
    gold_monitoring_directory="datamart/gold/model_monitoring/",
    charts_directory="datamart/gold/model_monitoring_charts/",
)

# show the dashboard; individual charts are in the same folder (for the deck)
display(Image(filename="datamart/gold/model_monitoring_charts/credit_model_2024_06_01/monitoring_dashboard.png"))